In [0]:
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd

test

In [0]:
n_features = 30
n_samples = 200_000
data = make_classification(n_samples = n_samples, n_features =n_features, random_state = 42, n_redundant = 10)


In [0]:
def train_test_split(data, test_size=0.33, seed=903):
  rng = np.random.default_rng(seed=seed)
  idx = rng.choice([True, False], size=data[0].shape[0], p=[1 - test_size, test_size])
  return (data[0][idx, :], data[1][idx]), (data[0][~idx, :], data[1][~idx])

def make_batch(dataset, batch_size = 512):
  data_batched = []
  for k in range(0, dataset[0].shape[0] // batch_size + 1):
    idx_left, idx_right = k * batch_size, min((k + 1) * batch_size, dataset[0].shape[0])
    data_batched.append((dataset[0][idx_left:idx_right, :], dataset[1][idx_left:idx_right]))
  assert (len(data_batched) - 1) * batch_size + data_batched[-1][0].shape[0] == dataset[0].shape[0]
  return data_batched

def random_normal_init(input_size, output_size, seed = 903):
  rng = np.random.default_rng(seed=seed)
  return rng.normal(size = (input_size, output_size)) / np.sqrt(input_size)

train, test = train_test_split(data)
train_batched = make_batch(train)
test_batched = make_batch(test)

In [0]:
w = random_normal_init(3, 1)
print(w)
print(w.shape)

In [0]:
rng = np.random.default_rng(seed=42)
X = rng.choice([-1, 1], size=(4, 3))
print(X)
print(X.shape)

In [0]:
EPS = 1e-12
def ll(y_true, y_pred):
  return -(np.matmul(y_true.T, np.log(y_pred + EPS)) + np.matmul((1 - y_true).T, np.log(1 - y_pred + EPS)))

def der_ll(y_true, y_pred):
  return np.mean(y_true - y_pred) / (np.matmul(y_pred.T, 1 - y_pred))
  
def sigmoid(g):
    return 1 / (1 + np.exp(-g))

In [0]:
print('normal X')
print(X, X.shape)
print('normal w')
print(w, w.shape)

print('\nX.T')
print(X.T, X.T.shape)
print('w squeezed')
print(np.squeeze(w), np.squeeze(w).shape)

# print('\n just dot(Xw)')
# print(np.dot(X,w))
print('\ndot(w.squeezed @ X.T)')
print(np.dot(np.squeeze(w), X.T))
print(sigmoid(np.dot(np.squeeze(w), X.T)), sigmoid(np.dot(np.squeeze(w), X.T)).shape)

In [0]:
class nn_1_layer_linear:
    def __init__(self, first_size: int, step: int) -> None:
        self.w0 = np.squeeze(random_normal_init(first_size, 1))
        self.b0 = random_normal_init(1, 1)[0][0]
        self.step = step
    
    def forward(self, x):
        weighted_sum = np.dot(self.w0, x.T) + self.b0
        y_pred = np.squeeze(sigmoid(weighted_sum))
        return y_pred

    def total_loss(self, y_true, y_pred):
        return ll(y_true, y_pred)

    def update_weights(self, x, y_true):
        y_pred = self.forward(x)
        self.w0 = self.w0 - self.step * np.dot(y_pred - y_true, x)
        self.b0 = self.b0 - self.step * np.mean(y_pred - y_true)
    
    def train(self, x, y_true,verbose = False):
        self.update_weights(x, y_true)
        if verbose:
            print(f"Loss: {self.total_loss(y_true, self.forward(x))}")


In [0]:
len(train_batched)

In [0]:
network = nn_1_layer_linear(n_features, step=0.01)

In [0]:
network.w0.shape

In [0]:
train_batched[0][0].T.shape

In [0]:
network.b0.shape

In [0]:
np.dot(network.w0, train_batched[0][0].T).shape

In [0]:
network = nn_1_layer_linear(n_features, step=0.01)
for epoch in range(50):
  for i in range(50):
    if i % 2 == 0:
      network.train(train_batched[i][0], train_batched[i][1], verbose = True)
    else:
      network.train(train_batched[i][0], train_batched[i][1])

In [0]:
from sklearn.metrics import roc_auc_score
roc_auc_score(test_batched[0][1], network.forward(test_batched[0][0]))

$$\frac{\partial{L}}{\partial{w_j}} = \frac{1}{m} (y_{pred} - y_{true}) \sigma(w^0_j x_i + b^0) $$
$$\frac{\partial{L}}{\partial{b}} = \frac{1}{m} \sum{(y^{i}_{pred} - y^{i}_{true})} $$
$$\frac{\partial{L}}{\partial{w^0_j}} = (y_{pred} - y_{true}) w_j \sigma(w^0_j x + b^0) (1 - \sigma(w^0_j x + b^0) ) x_i$$
$$\frac{\partial{L}}{\partial{b^0}} = (y_{pred} - y_{true}) w_j \sigma(w^0_j x + b^0) (1 - \sigma(w^0_j x + b^0) ) $$

In [0]:
class nn_2_layer:
    def __init__(self, first_size, second_size, step = 0.01):
        self.w0 = random_normal_init(first_size, second_size)
        self.b0 = random_normal_init(second_size, 1)

        self.w = random_normal_init(second_size, 1)
        self.b = random_normal_init(1, 1)[0][0]

        self.first_size = first_size
        self.second_size = second_size
        self.step = step
    
    def forward(self, x):
        y_pred = (self.w0 @ x) + self.b0
        y_pred = sigmoid(y_pred)
        y_pred = (y_pred @ self.w) + self.b
        return sigmoid(y_pred)

    def total_loss(self, y_true, y_pred):
        return ll(y_true, y_pred)

    def update_weights(self, x, y_ture):
        y_pred = self.forward(x)
        error = y_pred - y_ture


In [0]:
train_batched[]

In [0]:
n_features_t = 3
n_samples_t = 1000
data_t = make_classification(n_samples = n_samples_t, n_features =n_features_t, random_state = 42, n_redundant = 0)


train_t, test_t = train_test_split(data_t)
train_batched_t = make_batch(train_t)
test_batched_t = make_batch(test_t)

In [0]:
first_layer_ouput = train_batched_t[0][0] @ random_normal_init(3, 2) + np.squeeze(random_normal_init(2, 1))

second_layer_output = (1 / (1 + np.exp(-first_layer_ouput))) @ np.squeeze(random_normal_init(2, 1)) + random_normal_init(1, 1)[0][0]

(1 / (1 + np.exp(-second_layer_output)))
# second_layer_output

In [0]:
print('4 - 2 - 1')

print("\nFirst layer")
print(random_normal_init(3, 2), "shape: (3,2)")
print("Bias")
print(np.squeeze(random_normal_init(2, 1)), "shape: (2,1)")

print("\nSecond layer")
print(np.squeeze(random_normal_init(2, 1)), "shape: (2,1)")
print("Bias")
print(random_normal_init(1, 1)[0][0])